# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MOULEESHWARAN07/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata

print("HF_TOKEN found:", userdata.get("HF_TOKEN") is not None)

HF_TOKEN found: True


In [11]:
print(HF_TOKEN[:10])


hf_NJMAOoj


In [12]:
con.execute("DROP SECRET IF EXISTS hf")

con.execute(
    f"CREATE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

In [7]:
%pip -q install duckdb huggingface_hub

In [8]:
import os
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, table in TABLES.items():
    rows = con.sql(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(name, rows)

dim_clients 104
dim_content 519606


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily 78835655
fact_query_90d 2414248


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Unit of analysis
One row represents one content item (page) for one client.

### Time window
The input metrics are calculated over a trailing 90-day window. Labels must come from a future time period and should not overlap with the feature window.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Feature

Observable search and engagement metrics, including:
- ctr
- impressions_90d
- clicks_90d
- avg_position
- word_count
- engagement_rate
- scroll_rate

### Label
- is_declining_label
- trend_direction
- trend_pct

### Context
- content_id
- client_id
- content_type

### Excluded
- trend_direction and trend_pct because they are used to create the label.
- content_id and client_id are identifiers, not predictive features.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [13]:
con.sql(f"""
SELECT
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(*) AS total_rows
FROM {TABLES['fact_daily']}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_date,max_date,total_rows
0,2025-01-27,2026-06-30,78835655


In [14]:
con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS cnt
FROM {TABLES['fact_daily']}
GROUP BY 1,2,3
HAVING COUNT(*) > 1
LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,cnt
0,client_1a8bf67cad4ee525,content_3526ea2ae9cb48ee,2026-06-16,2
1,client_1a8bf67cad4ee525,content_a7928cdc075931a7,2026-06-16,2
2,client_def0955f7a377868,content_071de09fa78a164c,2026-06-15,2
3,client_def0955f7a377868,content_6bfa295a295c28e9,2026-06-22,2
4,client_8ddc46da5414ffd8,content_0c10db72295ef38f,2026-06-20,2


In [15]:
con.sql(f"""
SELECT
    AVG(CASE WHEN gsc_avg_position IS NULL THEN 1.0 ELSE 0 END) AS missing_position,
    AVG(CASE WHEN gsc_impressions IS NULL THEN 1.0 ELSE 0 END) AS missing_impressions,
    AVG(CASE WHEN gsc_clicks IS NULL THEN 1.0 ELSE 0 END) AS missing_clicks
FROM {TABLES['fact_daily']}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,missing_position,missing_impressions,missing_clicks
0,0.632527,0.001243,0.001243


The grain verification query returned duplicate rows for (client_hash_id, content_hash_id, report_date). This indicates that these columns alone do not uniquely identify every record, so an additional key or a more detailed grain definition is required.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### Data Limitations

- Clients have different amounts of historical data.
- Some GA4 and GSC values are unavailable and should be filtered using their availability flags.
- Trend-related fields cannot be used as features because they leak label information.
- Missing values are not completely random and often depend on content type or data availability.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.